In [2]:
import pandas as pd
import requests
import datetime

def obtener_historial_btc():
    url = "https://api.binance.com/api/v3/klines"
    desde_fecha = int(datetime.datetime(2020, 1, 1).timestamp() * 1000)
    ahora = int(datetime.datetime.now().timestamp() * 1000)
    
    todas_las_velas = []
    
    print("Iniciando descarga de datos desde Binance...")
    
    while desde_fecha < ahora:
        # Convertimos la fecha actual del bucle a formato legible para el print
        fecha_actual_legible = datetime.datetime.fromtimestamp(desde_fecha / 1000).strftime('%Y-%m-%d')
        print(f"-> Descargando lote de datos desde la fecha: {fecha_actual_legible}...")
        
        params = {
            "symbol": "BTCUSDT",
            "interval": "1d",
            "startTime": desde_fecha,
            "limit": 1000
        }
        
        try:
            response = requests.get(url, params=params, timeout=10).json()
        except Exception as e:
            print(f"Error de conexión: {e}. Reintentando...")
            continue
        
        if not response:
            break
            
        todas_las_velas.extend(response)
        desde_fecha = response[-1][0] + 1 
    
    columnas = ['Fecha', 'Open', 'High', 'Low', 'Close', 'Volume', 'CloseTime', 
                'QuoteAssetVolume', 'NumTrades', 'TakerBuyBase', 'TakerBuyQuote', 'Ignore']
    
    df = pd.DataFrame(todas_las_velas, columns=columnas)
    df['Fecha'] = pd.to_datetime(df['Fecha'], unit='ms')
    df['Close'] = df['Close'].astype(float)
    df = df[['Fecha', 'Close']]
    
    print(f"\n¡Éxito total! Se descargaron {len(df)} días de historial de precios.")
    return df

# Ejecutamos la función limpia
df_btc = obtener_historial_btc()
df_btc.head()

Iniciando descarga de datos desde Binance...
-> Descargando lote de datos desde la fecha: 2020-01-01...
-> Descargando lote de datos desde la fecha: 2022-09-26...
-> Descargando lote de datos desde la fecha: 2025-06-22...
-> Descargando lote de datos desde la fecha: 2026-07-16...

¡Éxito total! Se descargaron 2389 días de historial de precios.


,Fecha,Close
0,2020-01-02,6965.71
1,2020-01-03,7344.96
2,2020-01-04,7354.11
3,2020-01-05,7358.75
4,2020-01-06,7758.00


In [4]:
# 1. Creamos columnas adicionales para extraer el Año y el Mes de cada fecha
df_btc['Año'] = df_btc['Fecha'].dt.year
df_btc['Mes'] = df_btc['Fecha'].dt.month

# 2. Obtenemos el precio de apertura (inicio) y cierre (fin) de cada mes de cada año
df_mensual = df_btc.groupby(['Año', 'Mes']).agg(
    Precio_Inicio=('Close', 'first'),
    Precio_Fin=('Close', 'last')
).reset_index()

# 3. Calculamos el rendimiento porcentual de cada mes individual
df_mensual['Rendimiento_%'] = ((df_mensual['Precio_Fin'] - df_mensual['Precio_Inicio']) / df_mensual['Precio_Inicio']) * 100

# 4. Agrupamos los meses repetidos de todos los años para sacar la estadística histórica
analisis_estacional = df_mensual.groupby('Mes').agg(
    Rendimiento_Promedio=('Rendimiento_%', 'mean'), # <-- ¡El error estaba aquí, ya está corregido!
    Meses_Positivos=('Rendimiento_%', lambda x: (x > 0).sum()),
    Total_Años=('Rendimiento_%', 'count')
).reset_index()

# 5. Calculamos la probabilidad o ratio de éxito (cuántas veces subió)
analisis_estacional['Probabilidad_Subida_%'] = (analisis_estacional['Meses_Positivos'] / analisis_estacional['Total_Años']) * 100

# Mapeamos los números de los meses a sus nombres en español para que sea fácil de leer
nombres_meses = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio', 
                 7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
analisis_estacional['Mes'] = analisis_estacional['Mes'].map(nombres_meses)

# Redondeamos los números para que la tabla quede limpia
analisis_estacional = analisis_estacional.round(2)

# Mostramos el resultado final ordenado de mayor probabilidad a menor probabilidad
print("=== ANÁLISIS DE ESTACIONALIDAD DE BITCOIN (2020 - 2026) ===")
analisis_estacional.sort_values(by='Rendimiento_Promedio', ascending=False)

=== ANÁLISIS DE ESTACIONALIDAD DE BITCOIN (2020 - 2026) ===


,Mes,Rendimiento_Promedio,Meses_Positivos,Total_Años,Probabilidad_Subida_%
9,Octubre,15.85,5,6,83.33
6,Julio,11.68,6,7,85.71
0,Enero,8.60,4,7,57.14
10,Noviembre,7.95,3,6,50.00
1,Febrero,6.75,3,7,42.86
11,Diciembre,6.59,3,6,50.00
2,Marzo,4.36,5,7,71.43
3,Abril,3.12,4,7,57.14
8,Septiembre,-0.66,3,6,50.00
7,Agosto,-3.97,1,6,16.67
